# deepEmulator — Colab training (Drive-backed)

Trains a DDQN agent against a Game Boy cartridge headlessly on a Colab GPU. Bundles (`model.pt` + `metadata.json` + trajectories + metrics) land in your Drive at `MyDrive/deepEmulator/checkpoints/<cartridge>/<run>/`. A `latest.txt` marker is updated each save — so re-running this notebook after a Colab disconnect **automatically resumes** from where you left off.

**One-time Drive setup:**
```
MyDrive/deepEmulator/
  roms/PokemonRed.gb           # legally dumped, sha1 ea9bcae617fdf159b045185467ae58b2e4a48b9a
  states/init.state            # PyBoy save-state (curriculum start point)
  checkpoints/                 # created on first run
```

In [ ]:
# 1. GPU check (Colab pattern from more-than-words)
!nvidia-smi -L || echo 'no GPU — runtime > change runtime type > GPU'

In [ ]:
# 2. Install deepEmulator (replace with your fork URL once pushed)
!pip install -q git+https://github.com/juangarassino/deepEmulator.git

In [ ]:
# 3. Sanity check imports + GPU available to torch
import torch, deepEmulator
print('deepEmulator', deepEmulator.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

In [ ]:
# 4. Train. Mounts Drive, resolves paths under MyDrive/deepEmulator/, and
#    auto-resumes from the latest run for this cartridge if one exists.
#    Re-run this cell after a Colab disconnect — it picks up where it left off.
from deepEmulator.training.colab_train import run_in_colab

run_in_colab(
    cartridge='POKEMON RED',
    rom='deepEmulator/roms/PokemonRed.gb',
    init_state='deepEmulator/states/init.state',
    steps=100_000,
    max_episode_steps=2048,
    save_every=10_000,
    resume=True,
)

In [ ]:
# 5. Inspect metrics.tsv from the latest run
import pandas as pd, glob, os
from pathlib import Path
root = Path('/content/drive/MyDrive/deepEmulator/checkpoints/pokemon_red')
latest = (root / 'latest.txt').read_text().strip() if (root / 'latest.txt').exists() else sorted(glob.glob(str(root / '*')))[-1]
print('run:', latest)
df = pd.read_csv(os.path.join(latest, 'metrics.tsv'), sep='\t')
df.tail()

## Pulling the bundle locally

The `latest.txt` marker in `MyDrive/deepEmulator/checkpoints/<cartridge>/` always points at the most recent run. Two ways to pull:

```bash
# Option A: rclone (recommended)
rclone copy gdrive:deepEmulator/checkpoints/pokemon_red ./checkpoints/pokemon_red

# Option B: Drive desktop client — just point your local checkpoints/ at the mounted Drive folder.

deepemu-play --cartridge "POKEMON RED" --ckpt $(cat ./checkpoints/pokemon_red/latest.txt) --visible --arrows
```
(`deepemu-play` is wired in P4; `--arrows` in P5/P10.)